# Chapter 2 — Key Learnings

This notebook contains my main takeaways, definitions, and conceptual notes
from **Chapter 2 - Working with Text Data** of *'Build a Large Language Model (From Scratch)'* book by Sebastian Raschka.

## Chapter Objective

Transform raw text into batched numerical vectors that can be processed by
a GPT-like transformer.

## Converting raw text into GPT input embeddings

The core idea is:  

> **The tokenizer** produces token IDs,  
**the embedding layers** turn IDs into vectors, and  
**token plus positional embeddings** become the input to the transformer.

The process is:  
```
Raw text  
→ token IDs  
→ input/target batches  

Input token IDs  
→ token embeddings ───────────┐
                              ├→ combined input embeddings  (= transformer input)
Position IDs  
→ positional embeddings ──────┘ 
```

### A mental code template for converting raw text into GPT input embeddings
**1. Create a DataLoader that tokenizes the text and produces input/target batches** 
```python 
dataloader = create_dataloader_v1(...)
```

**2. Retrieve one batch**  
```python
inputs, targets = next(iter(dataloader))
```

**3. Create token embedding layer**  
```python
token_embedding_layer = torch.nn.Embedding(
    vocab_size,
    embedding_dim
)
```

**4. Convert token IDs to vectors**  
```python
token_embeddings = token_embedding_layer(inputs)
```

**5. Create positional embedding layer**  
```python
pos_embedding_layer = torch.nn.Embedding(
    context_length,
    embedding_dim
)
```

**6. Create position IDs**  
```python
position_ids = torch.arange(context_length)
```

**7. Convert position IDs to vectors**  
```python
pos_embeddings = pos_embedding_layer(position_ids)
```

**8. Combine token and position information**  
```python
input_embeddings = token_embeddings + pos_embeddings
```

## The most important shapes to remember
**Token IDs:** [batch_size, sequence_length]

**Token embeddings:** [batch_size, sequence_length, embedding_dim]

**Position embeddings:** [sequence_length, embedding_dim]  
(The positional embeddings have no batch dimension. PyTorch broadcasts the
same positional embeddings across every sequence in the batch.)  

**Final input embeddings:** [batch_size, sequence_length, embedding_dim]

In [ ]:
# The actual code

import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader


vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
raw_text = "Your sample text goes here..."
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
   stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs (i.e. Token IDs):\n", inputs)
print("\nTargets:\n", targets)
print("\nInputs shape:", inputs.shape) #  torch.Size([8, 4])
print("Targets shape:", targets.shape)


### Inputs and targets for next-token prediction

The target sequence is the input sequence shifted one token forward (we use a **sliding window approach** on tokenized data):

Input:  [token₁, token₂, token₃, token₄]  
Target: [token₂, token₃, token₄, token₅]

At every position, the model learns to predict the next token.

- Given `token₁`, predict `token₂`
- Given `token₁, token₂`, predict `token₃`
- Given `token₁, token₂, token₃`, predict `token₄`
- Given `token₁, token₂, token₃, token₄`, predict `token₅`

The targets are used later to calculate the training loss. Only the input token IDs are passed through the embedding layers.

```
GPTDatasetV1 class  
→ creates individual input/target pairs  

DataLoader class
→ groups those pairs into batches  

create_dataloader_v1 function  
→ convenience function that creates both and returns the DataLoader
```

In [ ]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape) # torch.Size([8, 4, 256])

# For a GPT model’s absolute embedding approach, we just need to create another embedding layer that 
# has the same embedding dimension as the token_embedding_ layer:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape) # torch.Size([4, 256])

input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape) # torch.Size([8, 4, 256])

## Key Takeaways

- LLMs cannot process raw text directly; text must be converted into numerical
  representations.
- A tokenizer converts text into tokens and token IDs.
- BPE represents text using reusable word and subword units.
- A sliding window creates input and target sequences for next-token prediction.
- `GPTDatasetV1` creates examples, while `DataLoader` groups them into batches.
- `torch.nn.Embedding` performs a trainable lookup from token IDs to vectors.
- Token embeddings represent token identity.
- Positional embeddings represent token order.
- Adding token and positional embeddings produces the input to the transformer.

## Q/As

- **How are embedding weights updated during pretraining?**  
  Backpropagation adjusts them to reduce the next-token prediction loss.

- **How does stride affect sample diversity and overfitting?**  
  A smaller stride creates more overlapping samples, which increases data reuse but may increase overfitting.

- **Why are token and positional embeddings added rather than concatenated?**  
  Addition preserves the embedding size while combining token identity and position.

- **How will attention use the final input embeddings?**  
  Attention transforms them into context-aware representations by relating each token to other tokens in the sequence.